[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BradSegal/OxfordADH2026PythonBootcamp/blob/main/solutions/06_L5_dictionaries.ipynb)

In [ ]:
# Setup: run this cell first. It installs the course helpers (about 20 seconds).
import importlib.util, subprocess, sys

if importlib.util.find_spec("adh_python") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/BradSegal/OxfordADH2026PythonBootcamp.git"],
        check=True,
    )
print("Course helpers ready.")

# Level 5: Dictionaries and patient records

A list preserves order, but a patient's record isn't really a list. You
don't look up "the fourth thing about this patient", you look up their bed,
their consultant, or their allergies by name. A **dictionary** (`dict` in
Python) is a data structure that stores values with names in the same way a
patient's chart will have a label associated with each value. Almost all
health data you interact with in Python from a single set of observations to
an entire ward is composed of some combination of dictionaries and lists.

**By the end of this level you can:**
- read, update and add entries in a patient record;
- handle a field that might be missing, using `in` and `.get`;
- reach inside nested data, such as the observations within a record; and
- loop over a ward of patients to filter and count them.

This level should take you about 30 minutes. As before, write your answer
where you see `...` and then run the cell to check it.

In [ ]:
from adh_python import data
from adh_python.checks import level5 as check

## A dictionary is a chart

A dictionary is written with curly brackets. Each entry is a **key** (the
label, usually text in quotes), a colon, and a **value** (the entry itself).
Entries are separated by commas.

In [ ]:
patient = {
    "name": "Jordan Nguyen",
    "hospital_number": "RTH-102938",
    "ward": "5B",
    "bed": 12,
}
print(patient["ward"])
print(patient["bed"] + 1)

Square brackets with a key read a value, so `patient["ward"]` is `"5B"`. The
values all retain their type so `patient["bed"]` is a number and can be used
in arithmetic operations.

The same square brackets to the left of a `=` modify the record. If the key
exists the value is **replaced**; if it doesn't, a new entry is **added**. A
dictionary can never have the same key twice.

In [ ]:
patient["bed"] = 9  # replaces the old bed number
patient["pronouns"] = "they/them"  # adds a new entry
print(patient)

## Problem 5.1: Predict

**Before you run anything**, predict the number the code will print.

`len` of a dictionary counts the number of keys.

```python
patient = {"name": "Jordan Nguyen", "ward": "5B", "bed": 12}
patient["bed"] = 14
patient["consultant"] = "Dr Priya Patel"
print(len(patient))
```

Store your prediction as a number in `prediction`. Run the check, then run
the code itself in the cell after.

In [ ]:
prediction = 4
check.predict_record_length(prediction)

In [ ]:
patient = {"name": "Jordan Nguyen", "ward": "5B", "bed": 12}
patient["bed"] = 14
patient["consultant"] = "Dr Priya Patel"
print(len(patient))
print(patient)

## A fuller record

The course helpers include Jordan's full record. `data.sample_patient()`
gives you a fresh copy each time, so you can change it freely. Print it and
look at the shape: some values are text, some are numbers, some are lists,
and `"observations"` is a dictionary of its own.

In [ ]:
patient = data.sample_patient()
print(patient.keys())
print(patient["consultant"])
print(patient["allergies"])

`patient.keys()` lists every key and is a quick way of checking what is
available in a record, as well as confirming the exact spelling. Keys are
exact: `"Ward"` is not the same as `"ward"`.

## Problem 5.2: Modify

Jordan has been moved from ward 5B to **ward 7A, bed 3** and is under the
care of **Dr Aisha Rahman**. The code below moves them to bed 14 on the same
ward. Update the code to reflect the full move. The code should update the
actual record; the last line only prints it.

In [ ]:
patient = data.sample_patient()
patient["ward"] = "7A"
patient["bed"] = 3
patient["consultant"] = "Dr Aisha Rahman"
print(
    f"{patient['name']}: ward {patient['ward']}, bed {patient['bed']}, under {patient['consultant']}"
)
check.transfer(patient)

Inside an f-string that is already using double quotes, you can write the
key in single quotes like `patient['name']` to ensure Python knows where the
end of the text is.

## When a key might be missing

Not every record has every field. Jordan's record doesn't have a
`"next_of_kin"` key and asking for one crashes the program with a
**KeyError**:

```
patient["next_of_kin"]
KeyError: 'next_of_kin'
```

The error tells you the name of the key that it couldn't find which is
usually all you need to fix it. There are two ways of asking for something
safely.

`key in patient` is `True` if the key exists and `False` if it does not. The
same `in` also works on a list, where it asks if a value is in the list.

In [ ]:
print("allergies" in patient)
print("next_of_kin" in patient)
print("Penicillin" in patient["allergies"])

`patient.get(key, default)` gives the value if the key exists, or the
default if it doesn't, without an error.

In [ ]:
print(patient.get("pronouns", "not recorded"))
print(patient.get("next_of_kin", "not recorded"))

A default is a decision, so decide carefully. A missing list of outstanding
tasks can safely imply "none", a missing allergy cannot. "Not yet asked" and
"no known drug allergies" are different facts and pretending they are the
same is a safety risk. `.get` will also silently return the default if you
misspell the key when square brackets would have alerted you immediately.

## Problem 5.3: Make, an allergy check

A good allergy record names the drug, the reaction and when it happened
(NICE guideline CG183). A penicillin allergy, though, is commonly recorded
as the whole class, because it applies to every penicillin. Jordan's record
says `"Penicillin"`, and amoxicillin is a penicillin, so the drug's own name
will not appear in the allergy list. Checking a new prescription therefore
takes two steps: look up the drug's class, then see whether that class is
in the patient's allergies.

The dictionary `drug_classes` is a small lookup table of drug name to class.
Write two lines:

1. store the class of `new_drug` in `drug_class` by looking it up in
   `drug_classes`, then
2. set `alert` to `True` if `drug_class` is in the patient's allergies or
   `False` if not. Make use of `in` and do not type out `True` or `False`.

Real prescribing systems check far more than this (some cephalosporins for
example need to be cautious with penicillin allergy). The table presented
here is a didactic simplification.

In [ ]:
drug_classes = {
    "amoxicillin": "Penicillin",
    "flucloxacillin": "Penicillin",
    "co-amoxiclav": "Penicillin",
    "clarithromycin": "Macrolide",
    "doxycycline": "Tetracycline",
}
patient = data.sample_patient()
new_drug = "amoxicillin"
drug_class = drug_classes[new_drug]
alert = drug_class in patient["allergies"]
print(f"{new_drug} is a {drug_class}. Allergy alert: {alert}")
check.allergy_check(drug_classes, patient["allergies"], new_drug, drug_class, alert)

Now change `new_drug` to `"clarithromycin"` and re-run the cell. Jordan is
in fact on clarithromycin, a macrolide is a common choice for pneumonia if a
patient is allergic to penicillin.

Note the use of square brackets for `drug_classes[new_drug]` and not `.get`.
If someone attempts to prescribe a drug the table is unaware of you want the
check to throw an error. If we used `.get` with a default, the check would
silently pass an allergy check for an unknown drug.

## Looping over a dictionary

A `for` loop over a dictionary iterates over the keys. The `.values()`
function returns the values, while the `.items()` function iterates over the
key and value simultaneously which is typically preferred. We can get the
latest observations for Jordan as a dictionary by calling
`data.latest_observations(patient)`.

In [ ]:
latest = data.latest_observations(patient)
for name, value in latest.items():
    print(f"{name}: {value}")

`for name, value in latest.items():` gives two loop variables at once, the
first is the key and the second its value.

## Nested data and a ward

A value can itself be a dictionary or list. Jordan's `"observations"` is a
dictionary with values that are lists. Each value is a list of readings from
each round of observations, with the oldest first. You access each bit one
step at a time, from left to right.

In [ ]:
observations = patient["observations"]
print(observations["pulse"])
print(patient["observations"]["pulse"][-1])  # the most recent pulse

A **ward** is a list of patient dictionaries. `data.ward(seed=2026)` gives
everyone the same patients. Each patient has an `"observations"` dictionary
which contains a current set of observations.

Look closely at the shape, because it is different to Jordan's. Unlike
Jordan's record where each value is a list of readings, each value here is a
single reading. `patient["observations"]["pulse"]` is a single number and
does not require a `[-1]`.

In [ ]:
patients = data.ward(seed=2026)
print(len(patients))
print(patients[0])
for patient in patients:
    print(patient["bed"], patient["name"], patient["observations"]["pulse"])

## Problem 5.4: Make, who is on oxygen?

The nurse in charge wants to know which patients are on supplemental oxygen.
Create a list `oxygen_beds` with the **bed numbers** of all patients whose
`"on_oxygen"` observation is `True` in the order they appear on the ward.

Start with an empty list, iterate over `patients`, use `.append` for each
patient on oxygen. `on_oxygen` is already `True` or `False`, so can directly
follow `if`.

In [ ]:
patients = data.ward(seed=2026)
oxygen_beds = []
for patient in patients:
    if patient["observations"]["on_oxygen"]:
        oxygen_beds.append(patient["bed"])
print(f"On oxygen: beds {oxygen_beds}")
check.oxygen_beds(patients, oxygen_beds)

## Problem 5.5: Make, counting by diagnosis

For the bed manager, let's count how many patients have each diagnosis.
Create a dictionary `counts` where the keys are the diagnoses and the value
is the number of patients with that diagnosis.

The first time you meet a diagnosis it is not in `counts` yet so
`counts[diagnosis] += 1` would fall over with a KeyError. This is the job
`.get` was made for, `counts.get(diagnosis, 0)` gives you the count so far
or 0 if it hasn't been seen. You can then add 1 to that and save it back to
`counts[diagnosis]`.

In [ ]:
patients = data.ward(seed=2026)
counts = {}
for patient in patients:
    diagnosis = patient["diagnosis"]
    counts[diagnosis] = counts.get(diagnosis, 0) + 1
for diagnosis, count in counts.items():
    print(f"{count} x {diagnosis}")
check.diagnosis_counts(patients, counts)

Here a default of 0 is exactly what you want, a diagnosis you haven't seen
yet truly does have a count of 0.

## Problem 5.6: Fix the bug

New observations have just been taken for one of the patients. Their bed
number is in `target_bed` and they're now on supplemental oxygen with an
SpO2 of `new_spo2`. This code is intended to record that in their
observations, however, while it runs without an error it changes the wrong
record. Run the code, read what it prints, and then find and fix the error.

In [ ]:
patients = data.ward(seed=2026)
target_bed = 7
new_spo2 = 94
for patient in patients:
    if patient["bed"] == target_bed:
        print("Found:", patient["name"])
        patient["observations"]["on_oxygen"] = True
        patient["observations"]["spo2"] = new_spo2
        print("Updated:", patient["name"])
check.oxygen_started(patients, target_bed, new_spo2)

The two update lines sat outside the `if` statement and so after the loop
had finished. `patient` at this point is the last patient the loop saw,
i.e., the last bed on the ward, and so this patient's record was changed and
not the target patient. This was a clue due to the 'Found' and 'Updated'
names being printed as different names. Indenting the lines moves them
within the if statement so they only run if the patient in `target_bed` is
reached. This is the code equivalent to writing notes on the wrong patient's
chart. There is no error produced here so check who you updated!

## Before you move on

- A dictionary stores values under keys. Square brackets read, replace or
  add an entry.
- `in` is used to check if a key exists, `.get(key, default)` is used to
  read a key that may not exist. A default is a clinical decision, not just
  a method of preventing an error.
- Nested data is read one step at a time:
  `patient["observations"]["spo2"]`.
- To find a patient in a ward, loop and test a field such as the bed
  number, and do the work inside the `if`.

**Up next:** Level 6, functions, where you'll convert these steps to
reusable tools and create a NEWS2 calculator.